# Imports

In [19]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
from typing import List
from torch.optim import Adam
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, random_split, DataLoader
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error

# Globals

In [20]:
device = 'cuda'

# Dataset

In [21]:
class WeatherDatasetSeasonal(Dataset):
    def __init__(self, csv_path, window_size=30):
        super().__init__()
        self.df = pd.read_csv(csv_path).dropna()
        X_df: pd.DataFrame = self.df[['BASEL_cloud_cover', 'BASEL_humidity', 'BASEL_pressure', 'BASEL_global_radiation', 'BASEL_precipitation', 'BASEL_sunshine']]
        X_np = X_df.to_numpy(np.float32)
        
        Y_df = self.df[['BASEL_temp_mean', 'BASEL_temp_min', 'BASEL_temp_max']]
        Y_np = Y_df.to_numpy(np.float32)
        
        new_X = []
        new_Y = []
        for r in range(window_size, X_np.shape[0]):
            new_X.append(X_np[r - window_size: r])
            new_Y.append(Y_np[r])
            
        self.X = torch.from_numpy(np.array(new_X))
        self.Y = torch.from_numpy(np.array(new_Y))
        
        print(self.X.shape, self.Y.shape)
        
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]
    
    def __len__(self):
        return len(self.X)

In [22]:
class WeatherDatasetNonSeasonal(Dataset):
    def __init__(self, csv_path, window_size=30):
        super().__init__()
        self.df = pd.read_csv(csv_path).dropna()
        X_df: pd.DataFrame = self.df[['BASEL_cloud_cover', 'BASEL_humidity', 'BASEL_pressure', 'BASEL_precipitation']]
        X_np = X_df.to_numpy(np.float32)
        
        Y_df = self.df[['BASEL_temp_mean', 'BASEL_temp_min', 'BASEL_temp_max']]
        Y_np = Y_df.to_numpy(np.float32)
        
        new_X = []
        new_Y = []
        for r in range(window_size, X_np.shape[0]):
            new_X.append(X_np[r - window_size: r])
            new_Y.append(Y_np[r])
            
        self.X = torch.from_numpy(np.array(new_X))
        self.Y = torch.from_numpy(np.array(new_Y))
        
        print(self.X.shape, self.Y.shape)
        
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]
    
    def __len__(self):
        return len(self.X)

In [23]:
seasonal_dataset = WeatherDatasetSeasonal('dataset/weather_prediction_dataset.csv')
non_seasonal_dataset = WeatherDatasetNonSeasonal('dataset/weather_prediction_dataset.csv')

torch.Size([3624, 30, 6]) torch.Size([3624, 3])
torch.Size([3624, 30, 4]) torch.Size([3624, 3])


# Model

In [24]:
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

In [25]:
class GRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        return self.fc(out)

# Metrics Calculation

In [26]:
def calc_metrics(Y_true: torch.Tensor, Y_pred: torch.Tensor):
    Y_pred_np = Y_pred.detach().cpu().numpy()
    Y_true_np = Y_true.detach().cpu().numpy()
    
    mae = mean_absolute_error(Y_true_np, Y_pred_np)
    rmse = root_mean_squared_error(Y_true_np, Y_pred_np)
    mse = mean_squared_error(Y_true_np, Y_pred_np)
    
    return mae, rmse, mse

# Inference Loop

In [27]:
def test(model, test_loader):
    preds = []
    targets = []

    with torch.no_grad():
        for X, Y in test_loader:
            X = X.to(device)
            preds.append(model(X).detach().cpu())
            targets.append(Y)

    Y_pred = torch.cat(preds)
    Y_true = torch.cat(targets)
    mae, rmse, mse = calc_metrics(Y_true, Y_pred)
    print(f'MAE on test dataset = {mae}\nRMSE on test dataset = {rmse}\nMSE on test dataset = {mse}')

In [28]:

def run(model, dataset, seed, non_train_size, epochs = 100, lr = 1e-3, print_epochs = False):
    torch.manual_seed(seed=seed)
    np.random.seed(seed=seed)

    train_dataset, test_dataset, val_dataset = random_split(dataset, [1 - non_train_size, non_train_size / 2, non_train_size / 2], generator=torch.Generator().manual_seed(seed))
    train_loader = DataLoader(train_dataset, shuffle=True, batch_size=32)
    test_loader = DataLoader(test_dataset)
    val_loader = DataLoader(val_dataset)
    
    optimiser = Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_loss_ot = []
    val_loss_ot = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", disable=not print_epochs)
        
        for inputs, targets in pbar:
            inputs = inputs.to(device)
            targets = targets.to(device)

            optimiser.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            loss.backward()
            optimiser.step()

            total_loss += loss.item()
            
        avg_train_loss = total_loss / len(train_loader)
        train_loss_ot.append(avg_train_loss)
        if print_epochs:
            print(f'Avg Training Loss = {avg_train_loss}')
        
        if epoch % 50 == 0:
            model.eval()
            test(model, val_loader)
            print()
    
    model.eval()
    return train_loader, test_loader, model, train_loss_ot, val_loss_ot

# Testing

## Seasonal, LSTM

In [29]:
train_loader, test_loader, model, train_loss_ot, val_loss_ot = run(LSTM(input_size=6, hidden_size=64, num_layers=2, output_size=3).to(device), seasonal_dataset, 321, 0.7)

MAE on test dataset = 2.811962127685547
RMSE on test dataset = 3.5497825145721436
MSE on test dataset = 12.695887565612793

MAE on test dataset = 2.582240343093872
RMSE on test dataset = 3.3255608081817627
MSE on test dataset = 11.192352294921875



In [30]:
test(model, test_loader)

MAE on test dataset = 2.6777350902557373
RMSE on test dataset = 3.4617068767547607
MSE on test dataset = 12.117355346679688


## Non-Seasonal, LSTM

In [31]:
train_loader, test_loader, model, train_loss_ot, val_loss_ot = run(LSTM(input_size=4, hidden_size=64, num_layers=2, output_size=3).to(device), non_seasonal_dataset, 321, 0.7)

MAE on test dataset = 6.303372859954834
RMSE on test dataset = 7.5080108642578125
MSE on test dataset = 57.0109977722168

MAE on test dataset = 3.6136763095855713
RMSE on test dataset = 4.517392635345459
MSE on test dataset = 20.57634735107422



In [32]:
test(model, test_loader)

MAE on test dataset = 3.675273895263672
RMSE on test dataset = 4.627543926239014
MSE on test dataset = 21.64299964904785


## Seasonal, GRU

In [33]:
train_loader, test_loader, model, train_loss_ot, val_loss_ot = run(GRU(input_size=6, hidden_size=64, num_layers=2, output_size=3).to(device), seasonal_dataset, 321, 0.7)

MAE on test dataset = 2.7594032287597656
RMSE on test dataset = 3.4834766387939453
MSE on test dataset = 12.232741355895996

MAE on test dataset = 2.474658250808716
RMSE on test dataset = 3.1682565212249756
MSE on test dataset = 10.165616035461426



In [34]:
test(model, test_loader)

MAE on test dataset = 2.5299489498138428
RMSE on test dataset = 3.2759673595428467
MSE on test dataset = 10.878130912780762


## Non-Seasonal, GRU

In [35]:
train_loader, test_loader, model, train_loss_ot, val_loss_ot = run(GRU(input_size=4, hidden_size=64, num_layers=2, output_size=3).to(device), non_seasonal_dataset, 321, 0.7)

MAE on test dataset = 6.304046630859375
RMSE on test dataset = 7.508800506591797
MSE on test dataset = 57.023380279541016

MAE on test dataset = 3.282625913619995
RMSE on test dataset = 4.122755527496338
MSE on test dataset = 17.14312171936035



In [36]:
test(model, test_loader)

MAE on test dataset = 3.2722110748291016
RMSE on test dataset = 4.10875129699707
MSE on test dataset = 17.075227737426758


# Results

- RNN or Recurrent Neural Network is a multi layer network that feeds its output back into its inputs, to further process before predicting.
- When a RNN recurs incessently, its gradients that change weights and biases sometimes explode or diminish, as in they either become very large (> 1e5) or very small (< 1e-2).

|Model|Dataset|Split|MAE|RMSE|MSE|
|-----|-------|-----|---|----|---|
|LSTM|Seasonal|Validation|2.582|3.325|11.19|
|LSTM|Seasonal|Test|2.677|3.461|12.117|
|LSTM|Non-Seasonal|Validation|3.613|4.517|20.576|
|LSTM|Non-Seasonal|Test|3.675|4.627|21.642|
|GRU|Seasonal|Validation|2.474|3.168|10.165|
|GRU|Seasonal|Test|2.529|3.275|10.878|
|GRU|Non-Seasonal|Validation|3.282|4.122|17.143|
|GRU|Non-Seasonal|Test|3.272|4.108|17.075|

### Seasonal, GRU gives the best results.